# 06 | Transferability, residual diagnostics and final reporting
Evaluate how the selected model degrades when transferred to another region/year. Use the validation residual 90th percentile as a descriptive error envelope, not a formal conformal prediction interval: the same validation set selected the model, and rows are correlated. Test empirical coverage of that envelope; there is no guaranteed 90% coverage.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

In [ ]:
pred=pd.read_csv(ROOT/'outputs'/'heldout_predictions.csv')
scores=pd.read_csv(ROOT/'outputs'/'model_metrics.csv')
selected=joblib.load(ROOT/'models'/'retrieval.joblib')['selected']
d=pred[pred.model==selected].copy(); val=d[d.split=='validation']
assert len(val)>0
radius=float(np.quantile(np.abs(val.prediction-val.sm),.90,method='higher'))
rows=[]
for split in ['validation','spatial_test','temporal_test','joint_test']:
    z=d[d.split==split]; rows.append(dict(split=split,**sw.metrics(z.sm,z.prediction),radius=radius,empirical_coverage=float((np.abs(z.prediction-z.sm)<=radius).mean()) if len(z) else np.nan,months=z.month.nunique(),cells=z[['row','col']].drop_duplicates().shape[0]))
envelope=pd.DataFrame(rows); display(envelope.round(4)); envelope.to_csv(ROOT/'outputs'/'residual_envelope.csv',index=False)

## Sensitivity to coverage and crop fraction
Keep the fitted model fixed. Do not select a new threshold on holdout performance. Report zero or small sample sizes. Coverage sensitivity only applies to available eligible rows (≥75% support). Crop fraction is derived from 2021 land cover, not annual crop maps.

In [ ]:
rows=[]
for split in ['spatial_test','temporal_test','joint_test']:
    z=d[d.split==split]
    for cut in [.75,.85,.95]:
        zz=z[z.coverage>=cut]; rows.append(dict(split=split,criterion='coverage',threshold=cut,**sw.metrics(zz.sm,zz.prediction)))
    for cut in [.0,.3,.5]:
        zz=z[z.crop_fraction>=cut]; rows.append(dict(split=split,criterion='crop_fraction',threshold=cut,**sw.metrics(zz.sm,zz.prediction)))
sensitivity=pd.DataFrame(rows); display(sensitivity.round(4)); sensitivity.to_csv(ROOT/'outputs'/'threshold_sensitivity.csv',index=False)

## Monthly errors and seasonal baseline
Group the temporal and joint test residuals by month. Compare the seasonal baseline to fusion models; a seasonal NDVI proxy may beat SAR. Monthly aggregation and reference footprint smoothing limit what can be concluded.

In [ ]:
rows=[]
for (model,split,month),z in pred[pred.split.isin(['temporal_test','joint_test'])].groupby(['model','split','month']): rows.append(dict(model=model,split=split,month=month,**sw.metrics(z.sm,z.prediction)))
monthly=pd.DataFrame(rows); monthly.to_csv(ROOT/'outputs'/'monthly_test_metrics.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,split in zip(axes,['temporal_test','joint_test']):
    for name in dict.fromkeys(['train_mean','seasonal_ridge','sar_ridge','fusion_ridge',selected]):
        z=monthly[(monthly.model==name)&(monthly.split==split)].sort_values('month'); ax.plot(z.month,z.rmse,marker='o',label=name)
    ax.tick_params(axis='x',rotation=55); ax.set(title=split,ylabel='RMSE (m³/m³)'); ax.legend(fontsize=7)
fig.savefig(ROOT/'figures'/'06_monthly_rmse.png',dpi=150); plt.show()

**Final checkpoint:** report observed sample sizes, baseline comparisons, spatial and temporal errors, empirical residual coverage and map support. A successful result is a defensible assessment, not necessarily high R². No field validation, no independent fine-scale moisture labels, no root-zone inference. The source product is [SMAP L3 enhanced 9 km](https://developers.google.com/earth-engine/datasets/catalog/NASA_SMAP_SPL3SMP_E_006); [Sentinel-1](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S1_GRD), [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), [WorldCover](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200).